# Pandas: map · transform · pivot_table · cut · qcut

En este notebook cubrimos cuatro herramientas de pandas para transformar y resumir datos:

1. `map` — reemplazar valores de una Serie con un diccionario o función
2. `transform` — agregar estadísticas de grupo sin colapsar el DataFrame
3. `pivot_table` — resúmenes en formato tabla de doble entrada
4. `cut` / `qcut` — discretizar variables numéricas en rangos

Usaremos un dataset de ventas de empleados a lo largo de todo el notebook.

In [1]:
import pandas as pd
import numpy as np

data = {
    'empleado':   ['Ana García', 'Luis Pérez', 'Carmen Ruiz', 'Carlos López', 'Ana García',
                   'Luis Pérez', 'Carmen Ruiz', 'Carlos López', 'Ana García', 'Luis Pérez',
                   'Carmen Ruiz', 'Carlos López', 'Ana García', 'Luis Pérez', 'Carmen Ruiz'],
    'region':     ['Norte', 'Sur', 'Norte', 'Centro', 'Norte',
                   'Sur', 'Norte', 'Centro', 'Norte', 'Sur',
                   'Norte', 'Centro', 'Norte', 'Sur', 'Norte'],
    'categoria':  ['Electrónica', 'Ropa', 'Hogar', 'Electrónica', 'Ropa',
                   'Hogar', 'Electrónica', 'Ropa', 'Hogar', 'Electrónica',
                   'Ropa', 'Hogar', 'Electrónica', 'Ropa', 'Hogar'],
    'mes':        ['Enero', 'Enero', 'Enero', 'Enero', 'Febrero',
                   'Febrero', 'Febrero', 'Febrero', 'Marzo', 'Marzo',
                   'Marzo', 'Marzo', 'Abril', 'Abril', 'Abril'],
    'ventas':     [1500, 980, 2300, 750, 1800,
                   1100, 2100, 890, 2200, 1350,
                   1950, 600, 1700, 1250, 2400],
    'unidades':   [30, 45, 20, 15, 36,
                   50, 18, 22, 44, 60,
                   17, 12, 34, 55, 21],
    'nivel':      ['Senior', 'Junior', 'Senior', 'Junior', 'Senior',
                   'Junior', 'Senior', 'Junior', 'Senior', 'Mid',
                   'Senior', 'Junior', 'Mid', 'Mid', 'Senior']
}

df = pd.DataFrame(data)
df

,empleado,region,categoria,mes,ventas,unidades,nivel
0,Ana García,Norte,Electrónica,Enero,1500,30,Senior
1,Luis Pérez,Sur,Ropa,Enero,980,45,Junior
2,Carmen Ruiz,Norte,Hogar,Enero,2300,20,Senior
3,Carlos López,Centro,Electrónica,Enero,750,15,Junior
4,Ana García,Norte,Ropa,Febrero,1800,36,Senior
5,Luis Pérez,Sur,Hogar,Febrero,1100,50,Junior
6,Carmen Ruiz,Norte,Electrónica,Febrero,2100,18,Senior
7,Carlos López,Centro,Ropa,Febrero,890,22,Junior
8,Ana García,Norte,Hogar,Marzo,2200,44,Senior
9,Luis Pérez,Sur,Electrónica,Marzo,1350,60,Mid


In [ ]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   empleado   15 non-null     str  
 1   region     15 non-null     str  
 2   categoria  15 non-null     str  
 3   mes        15 non-null     str  
 4   ventas     15 non-null     int64
 5   unidades   15 non-null     int64
 6   nivel      15 non-null     str  
dtypes: int64(2), str(5)
memory usage: 972.0 bytes


`map` aplica una transformación **elemento por elemento** sobre una Serie.
Es la herramienta correcta para reemplazar o transformar valores de tres formas:

**Sintaxis:**
```python
serie.map(diccionario)   # reemplaza cada valor según el diccionario; valores ausentes → NaN
serie.map(funcion)       # aplica la función a cada elemento (lambda o función nombrada)
serie.map(serie)         # usa otra Serie como tabla de lookup (índice → valor)
```

> Si un valor no aparece en el diccionario o en el índice de la Serie, `map` devuelve `NaN`. Usá `.fillna()` después si necesitás cubrir esos casos.

In [3]:
# map con diccionario — reemplazar valores categóricos por códigos numéricos
# Úsalo cuando tenés un mapeo fijo y conocido de antemano

mes_a_num = {
    'Enero': 1,
    'Febrero': 2,
    'Marzo': 3,
    'Abril': 4
}

df['mes_num'] = df['mes'].map(mes_a_num)

display(df[['mes', 'mes_num']].drop_duplicates().sort_values('mes_num'))

,mes,mes_num
0,Enero,1
4,Febrero,2
8,Marzo,3
12,Abril,4


In [4]:
# ¿Qué pasa si falta un valor en el diccionario?
# map devuelve NaN para los valores no mapeados. Se resuelve con fillna()

mapa_incompleto = {'Enero': 1, 'Febrero': 2}  # 'Marzo' y 'Abril' no están

resultado = df['mes'].map(mapa_incompleto)
print(resultado.value_counts(dropna=False))   # muestra los NaN

# Solución: asignar un valor por defecto
resultado_limpio = df['mes'].map(mapa_incompleto).fillna(0).astype(int)
print(resultado_limpio.unique())

mes
NaN    7
1.0    4
2.0    4
Name: count, dtype: int64
[1 2 0]


In [5]:
# map con función nombrada — para lógicas que no caben en una línea

# Úsalo cuando la transformación tiene condiciones o pasos intermedios
def clasificar_venta(monto):
    if monto < 1000:
        return 'Baja'
    elif monto < 1800:
        return 'Media'
    else:
        return 'Alta'

df['categoria_venta'] = df['ventas'].map(clasificar_venta)

df[['empleado', 'mes', 'ventas', 'categoria_venta']].head(8)

,empleado,mes,ventas,categoria_venta
0,Ana García,Enero,1500,Media
1,Luis Pérez,Enero,980,Baja
2,Carmen Ruiz,Enero,2300,Alta
3,Carlos López,Enero,750,Baja
4,Ana García,Febrero,1800,Alta
5,Luis Pérez,Febrero,1100,Media
6,Carmen Ruiz,Febrero,2100,Alta
7,Carlos López,Febrero,890,Baja


In [6]:
# map con lambda — para transformaciones cortas en una sola línea

# Úsalo cuando la lógica es simple y no vale la pena definir una función aparte
df['ventas_fmt'] = df['ventas'].map(lambda x: f'$ {x:,.0f}')

df[['empleado', 'ventas', 'ventas_fmt']].head(6)

,empleado,ventas,ventas_fmt
0,Ana García,1500,"$ 1,500"
1,Luis Pérez,980,$ 980
2,Carmen Ruiz,2300,"$ 2,300"
3,Carlos López,750,$ 750
4,Ana García,1800,"$ 1,800"
5,Luis Pérez,1100,"$ 1,100"


In [7]:
# Limpiamos columnas auxiliares creadas en los ejemplos anteriores
df = df.drop(columns=['mes_num', 'categoria_venta', 'ventas_fmt'])
df.head(3)

,empleado,region,categoria,mes,ventas,unidades,nivel
0,Ana García,Norte,Electrónica,Enero,1500,30,Senior
1,Luis Pérez,Sur,Ropa,Enero,980,45,Junior
2,Carmen Ruiz,Norte,Hogar,Enero,2300,20,Senior


In [ ]:
# map con Serie de pandas — usar otra Serie como tabla de lookup
#
# La Serie actúa como diccionario: su índice son las claves y sus valores son los resultados.
# Es útil cuando el mapeo ya está en un objeto pandas (por ejemplo, viene de otro DataFrame).

bonos = pd.Series({
    'Junior': 5_000,
    'Mid':    8_000,
    'Senior': 12_000
})

df['bono'] = df['nivel'].map(bonos)

df[['empleado', 'nivel', 'bono']].drop_duplicates().sort_values('nivel')

,empleado,nivel,bono
1,Luis Pérez,Junior,5000
3,Carlos López,Junior,5000
9,Luis Pérez,Mid,8000
12,Ana García,Mid,8000
0,Ana García,Senior,12000
2,Carmen Ruiz,Senior,12000


---
## 2. `transform` vs `agg`

`transform` se usa **después de un `groupby`**. A diferencia de `agg`, no colapsa el DataFrame: devuelve una Serie del **mismo tamaño** que el original, donde cada fila recibe el valor calculado para su grupo.

| Método | Tamaño de salida | ¿Cuándo usarlo? |
|--------|------------------|-----------------|
| `agg`  | Una fila por grupo | Resumir |
| `transform` | Mismo que el original | Agregar contexto del grupo a cada fila |

**Sintaxis:**
```python
df.groupby('columna')['otra'].transform('mean')
df.groupby('columna')['otra'].transform(funcion)
```

---
## Repaso: `groupby` + `agg`

Antes de ver `transform`, repasamos `groupby` + `agg` porque `transform` se entiende mejor por contraste.

`agg` colapsa el DataFrame: devuelve **una fila por grupo**.

`transform` no colapsa: devuelve **una columna del mismo tamaño que el original**.

In [ ]:
# groupby + agg: una fila por grupo
# Ventas totales y promedio por empleado
df.groupby('empleado')['ventas'].agg(
    total='sum',
    promedio='mean'
).round(0)

,total,promedio
empleado,,
Ana García,7200,1800.0
Carlos López,2240,747.0
Carmen Ruiz,8750,2188.0
Luis Pérez,4680,1170.0


In [ ]:
# groupby + agg con múltiples columnas de agrupación
# Ventas totales por región y categoría
df.groupby(['region', 'categoria'])['ventas'].agg(
    total='sum',
    cantidad='count',
    maximo='max'
).reset_index()

,region,categoria,total,cantidad,maximo
0,Centro,Electrónica,750,1,750
1,Centro,Hogar,600,1,600
2,Centro,Ropa,890,1,890
3,Norte,Electrónica,5300,3,2100
4,Norte,Hogar,6900,3,2400
5,Norte,Ropa,3750,2,1950
6,Sur,Electrónica,1350,1,1350
7,Sur,Hogar,1100,1,1100
8,Sur,Ropa,2230,2,1250


In [ ]:
# Comparación agg vs transform
print('=== agg: una fila por empleado ===')
print(df.groupby('empleado')['ventas'].agg('mean').round(0))
print()
print('=== transform: una columna del mismo largo que df ===')
print(df.groupby('empleado')['ventas'].transform('mean').round(0))

=== agg: una fila por empleado ===
empleado
Ana García      1800.0
Carlos López     747.0
Carmen Ruiz     2188.0
Luis Pérez      1170.0
Name: ventas, dtype: float64

=== transform: una columna del mismo largo que df ===
0     1800.0
1     1170.0
2     2188.0
3      747.0
4     1800.0
5     1170.0
6     2188.0
7      747.0
8     1800.0
9     1170.0
10    2188.0
11     747.0
12    1800.0
13    1170.0
14    2188.0
Name: ventas, dtype: float64


## `transform` — Estadísticas de grupo en cada fila
¿Cuándo usar transform?
Cuando querés agregar una columna al DataFrame original con una estadística del grupo al que pertenece cada fila. Por ejemplo: media del grupo, normalización dentro del grupo.

In [ ]:
# Ejemplo 1: media de ventas por empleado agregada a cada fila
df['media_empleado'] = df.groupby('empleado')['ventas'].transform('mean').round(0)

df[['empleado', 'mes', 'ventas', 'media_empleado']].sort_values('empleado')

,empleado,mes,ventas,media_empleado
0,Ana García,Enero,1500,1800.0
4,Ana García,Febrero,1800,1800.0
8,Ana García,Marzo,2200,1800.0
12,Ana García,Abril,1700,1800.0
3,Carlos López,Enero,750,747.0
7,Carlos López,Febrero,890,747.0
11,Carlos López,Marzo,600,747.0
2,Carmen Ruiz,Enero,2300,2188.0
6,Carmen Ruiz,Febrero,2100,2188.0
10,Carmen Ruiz,Marzo,1950,2188.0


In [ ]:
# Ejemplo 2: ¿esta venta estuvo por encima del promedio del empleado?
df['sobre_media'] = df['ventas'] > df['media_empleado']

df[['empleado', 'mes', 'ventas', 'media_empleado', 'sobre_media']].sort_values('empleado')

,empleado,mes,ventas,media_empleado,sobre_media
0,Ana García,Enero,1500,1800.0,False
4,Ana García,Febrero,1800,1800.0,False
8,Ana García,Marzo,2200,1800.0,True
12,Ana García,Abril,1700,1800.0,False
3,Carlos López,Enero,750,747.0,True
7,Carlos López,Febrero,890,747.0,True
11,Carlos López,Marzo,600,747.0,False
2,Carmen Ruiz,Enero,2300,2188.0,True
6,Carmen Ruiz,Febrero,2100,2188.0,False
10,Carmen Ruiz,Marzo,1950,2188.0,False


In [ ]:
# Ejemplo 3: porcentaje que representa cada venta sobre el total de su empleado
total_empleado = df.groupby('empleado')['ventas'].transform('sum')
df['total_empleado'] = total_empleado
df['pct_del_empleado'] = (df['ventas'] / total_empleado * 100).round(1)

df[['empleado', 'mes', 'ventas', 'pct_del_empleado','total_empleado']].sort_values('empleado')

,empleado,mes,ventas,pct_del_empleado,total_empleado
0,Ana García,Enero,1500,20.8,7200
4,Ana García,Febrero,1800,25.0,7200
8,Ana García,Marzo,2200,30.6,7200
12,Ana García,Abril,1700,23.6,7200
3,Carlos López,Enero,750,33.5,2240
7,Carlos López,Febrero,890,39.7,2240
11,Carlos López,Marzo,600,26.8,2240
2,Carmen Ruiz,Enero,2300,26.3,8750
6,Carmen Ruiz,Febrero,2100,24.0,8750
10,Carmen Ruiz,Marzo,1950,22.3,8750


In [ ]:
# Ejemplo 4: normalización dentro del grupo (z-score por región)
def z_score(serie):
    return (serie - serie.mean()) / serie.std()

df['ventas_norm'] = df.groupby('region')['ventas'].transform(z_score).round(2)

df[['empleado', 'region', 'ventas', 'ventas_norm']].sort_values('region')

,empleado,region,ventas,ventas_norm
3,Carlos López,Centro,750,0.02
7,Carlos López,Centro,890,0.99
11,Carlos López,Centro,600,-1.01
0,Ana García,Norte,1500,-1.58
2,Carmen Ruiz,Norte,2300,0.98
4,Ana García,Norte,1800,-0.62
6,Carmen Ruiz,Norte,2100,0.34
8,Ana García,Norte,2200,0.66
10,Carmen Ruiz,Norte,1950,-0.14
12,Ana García,Norte,1700,-0.94


In [ ]:
df

,empleado,region,categoria,mes,ventas,unidades,nivel,bono,media_empleado,sobre_media,total_empleado,pct_del_empleado,ventas_norm
0,Ana García,Norte,Electrónica,Enero,1500,30,Senior,12000,1800.0,False,7200,20.8,-1.58
1,Luis Pérez,Sur,Ropa,Enero,980,45,Junior,5000,1170.0,False,4680,20.9,-1.16
2,Carmen Ruiz,Norte,Hogar,Enero,2300,20,Senior,12000,2188.0,True,8750,26.3,0.98
3,Carlos López,Centro,Electrónica,Enero,750,15,Junior,5000,747.0,True,2240,33.5,0.02
4,Ana García,Norte,Ropa,Febrero,1800,36,Senior,12000,1800.0,False,7200,25.0,-0.62
5,Luis Pérez,Sur,Hogar,Febrero,1100,50,Junior,5000,1170.0,False,4680,23.5,-0.43
6,Carmen Ruiz,Norte,Electrónica,Febrero,2100,18,Senior,12000,2188.0,False,8750,24.0,0.34
7,Carlos López,Centro,Ropa,Febrero,890,22,Junior,5000,747.0,True,2240,39.7,0.99
8,Ana García,Norte,Hogar,Marzo,2200,44,Senior,12000,1800.0,True,7200,30.6,0.66
9,Luis Pérez,Sur,Electrónica,Marzo,1350,60,Mid,8000,1170.0,True,4680,28.8,1.10


In [ ]:
# Limpiamos columnas auxiliares
df = df.drop(columns=['media_empleado', 'sobre_media', 'pct_del_empleado', 'ventas_norm', 'total_empleado'])

---
## 3. `pivot_table` — Tablas dinámicas

Una tabla dinámica resume datos organizándolos en filas y columnas según categorías. Es el equivalente en pandas de las tablas dinámicas de Excel — y produce el mismo resultado que un `groupby`, pero en un formato más legible para comparar categorías entre sí.

**Sintaxis:**
```python
pd.pivot_table(
    data,
    values='columna_a_agregar',
    index='columna_filas',
    columns='columna_columnas',   # opcional
    aggfunc='mean',
    fill_value=0,
    margins=True
)
```

In [ ]:
# Ejemplo 1: ventas totales por empleado y mes
pd.pivot_table(
    df,
    values='ventas',
    index='empleado',
    columns='mes',
    aggfunc='sum',
    fill_value=0
)

mes,Abril,Enero,Febrero,Marzo
empleado,,,,
Ana García,1700,1500,1800,2200
Carlos López,0,750,890,600
Carmen Ruiz,2400,2300,2100,1950
Luis Pérez,1250,980,1100,1350


In [ ]:
# Ejemplo 2: agregar totales con margins=True
pd.pivot_table(
    df,
    values='ventas',
    index='empleado',
    columns='mes',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

mes,Abril,Enero,Febrero,Marzo,Total
empleado,,,,,
Ana García,1700,1500,1800,2200,7200
Carlos López,0,750,890,600,2240
Carmen Ruiz,2400,2300,2100,1950,8750
Luis Pérez,1250,980,1100,1350,4680
Total,5350,5530,5890,6100,22870


In [ ]:
# Ejemplo 3: ventas totales por región y categoría
pd.pivot_table(
    df,
    values='ventas',
    index='region',
    columns='categoria',
    aggfunc='sum',
    fill_value=0
)

categoria,Electrónica,Hogar,Ropa
region,,,
Centro,750,600,890
Norte,5300,6900,3750
Sur,1350,1100,2230


In [ ]:
# Ejemplo 4: múltiples funciones de agregación
tabla = pd.pivot_table(
    df,
    values='ventas',
    index='empleado',
    aggfunc=['sum', 'mean', 'count']
)
tabla.columns = ['Total ventas', 'Promedio', 'Registros']
tabla

,Total ventas,Promedio,Registros
empleado,,,
Ana García,7200,1800.000000,4
Carlos López,2240,746.666667,3
Carmen Ruiz,8750,2187.500000,4
Luis Pérez,4680,1170.000000,4


In [ ]:
# Ejemplo 5: diferencia entre groupby y pivot_table sobre los mismos datos

# groupby → formato largo (una fila por grupo)
print('=== groupby ===')
print(df.groupby(['region', 'categoria'])['ventas'].sum().reset_index())

print()

# pivot_table → formato ancho (más fácil de leer comparando)
print('=== pivot_table ===')
print(pd.pivot_table(df, values='ventas', index='region',
                     columns='categoria', aggfunc='sum', fill_value=0))

=== groupby ===
   region    categoria  ventas
0  Centro  Electrónica     750
1  Centro        Hogar     600
2  Centro         Ropa     890
3   Norte  Electrónica    5300
4   Norte        Hogar    6900
5   Norte         Ropa    3750
6     Sur  Electrónica    1350
7     Sur        Hogar    1100
8     Sur         Ropa    2230

=== pivot_table ===
categoria  Electrónica  Hogar  Ropa
region                             
Centro             750    600   890
Norte             5300   6900  3750
Sur               1350   1100  2230


---
## 4. `cut` y `qcut` — Discretizar variables numéricas

Discretizar es convertir una variable numérica continua en categorías (rangos).
Es útil para EDA, para crear features categóricas, y para segmentar datos.

### `pd.cut` — rangos definidos por el usuario

Los límites los definís vos. Ideal cuando los rangos tienen un significado de negocio conocido.

```python
pd.cut(serie, bins=[límites], labels=['etiquetas'])
```

### `pd.qcut` — rangos por cuantiles (grupos de igual tamaño)

Pandas calcula los límites automáticamente para que cada grupo tenga la misma cantidad de observaciones.

```python
pd.qcut(serie, q=4, labels=['Q1','Q2','Q3','Q4'])
```

| | `cut` | `qcut` |
|---|---|---|
| Límites | Definidos por el usuario | Calculados por cuantiles |
| Ancho de rangos | Igual | Variable |
| Cantidad de obs. por grupo | Variable | Igual |
| Cuándo usarlo | Reglas de negocio conocidas | Percentiles, segmentación equitativa |

In [ ]:
# cut: clasificar ventas en rangos de negocio
df['rango_ventas'] = pd.cut(
    df['ventas'],
    bins=[0, 999, 1499, 1999, float('inf')],
    labels=['Baja', 'Media', 'Alta', 'Muy Alta']
)

df[['empleado', 'ventas', 'rango_ventas']].sort_values('ventas')

,empleado,ventas,rango_ventas
11,Carlos López,600,Baja
3,Carlos López,750,Baja
7,Carlos López,890,Baja
1,Luis Pérez,980,Baja
5,Luis Pérez,1100,Media
13,Luis Pérez,1250,Media
9,Luis Pérez,1350,Media
0,Ana García,1500,Alta
12,Ana García,1700,Alta
4,Ana García,1800,Alta


In [ ]:
# Distribución de ventas por rango
df['rango_ventas'].value_counts().sort_index()

rango_ventas
Baja        4
Media       3
Alta        4
Muy Alta    4
Name: count, dtype: int64

In [ ]:
# cut con N bins automáticos (pandas calcula los límites)
df['rango_auto'] = pd.cut(df['ventas'], bins=3, labels=['Bajo', 'Medio', 'Alto'])
df[['ventas', 'rango_auto']].sort_values('ventas')

,ventas,rango_auto
11,600,Bajo
3,750,Bajo
7,890,Bajo
1,980,Bajo
5,1100,Bajo
13,1250,Medio
9,1350,Medio
0,1500,Medio
12,1700,Medio
4,1800,Medio


In [ ]:
# qcut: dividir en cuartiles (grupos de igual tamaño)
df['cuartil'] = pd.qcut(
    df['ventas'],
    q=4,
    labels=['Q1 - Bajo', 'Q2', 'Q3', 'Q4 - Alto']
)

df[['empleado', 'ventas', 'cuartil']].sort_values('ventas')

,empleado,ventas,cuartil
11,Carlos López,600,Q1 - Bajo
3,Carlos López,750,Q1 - Bajo
7,Carlos López,890,Q1 - Bajo
1,Luis Pérez,980,Q1 - Bajo
5,Luis Pérez,1100,Q2
13,Luis Pérez,1250,Q2
9,Luis Pérez,1350,Q2
0,Ana García,1500,Q2
12,Ana García,1700,Q3
4,Ana García,1800,Q3


In [ ]:
# qcut garantiza igual cantidad de observaciones por grupo
df['cuartil'].value_counts().sort_index()

cuartil
Q1 - Bajo    4
Q2           4
Q3           3
Q4 - Alto    4
Name: count, dtype: int64

In [ ]:
# Ver los límites que calculó qcut
_, limites = pd.qcut(df['ventas'], q=4, retbins=True)
print('Límites de los cuartiles:')
for i in range(len(limites) - 1):
    print(f'  Q{i+1}: {limites[i]:.0f} — {limites[i+1]:.0f}')

Límites de los cuartiles:
  Q1: 600 — 1040
  Q2: 1040 — 1500
  Q3: 1500 — 2025
  Q4: 2025 — 2400


In [ ]:
# Comparación final: cut vs qcut sobre los mismos datos
resumen = pd.DataFrame({
    'ventas': df['ventas'],
    'cut (rangos iguales)':    pd.cut(df['ventas'],  bins=4, labels=['R1','R2','R3','R4']),
    'qcut (tamaños iguales)': pd.qcut(df['ventas'], q=4,    labels=['Q1','Q2','Q3','Q4'])
}).sort_values('ventas')

print(resumen)
print()
print('Distribución cut:')
print(resumen['cut (rangos iguales)'].value_counts().sort_index())
print()
print('Distribución qcut:')
print(resumen['qcut (tamaños iguales)'].value_counts().sort_index())

    ventas cut (rangos iguales) qcut (tamaños iguales)
11     600                   R1                     Q1
3      750                   R1                     Q1
7      890                   R1                     Q1
1      980                   R1                     Q1
5     1100                   R2                     Q2
13    1250                   R2                     Q2
9     1350                   R2                     Q2
0     1500                   R2                     Q2
12    1700                   R3                     Q3
4     1800                   R3                     Q3
10    1950                   R3                     Q3
6     2100                   R4                     Q4
8     2200                   R4                     Q4
2     2300                   R4                     Q4
14    2400                   R4                     Q4

Distribución cut:
cut (rangos iguales)
R1    4
R2    4
R3    3
R4    4
Name: count, dtype: int64

Distribución qcut:
qc

---
## Resumen

| Herramienta | ¿Qué hace? | Caso de uso típico |
|-------------|-----------|--------------------|
| `map` | Transforma cada valor de una Serie | Codificar categorías con un diccionario |
| `transform` | Estadística del grupo en cada fila | Agregar contexto del grupo sin colapsar el DataFrame |
| `pivot_table` | Tabla de doble entrada con agregaciones | Resúmenes comparativos entre categorías |
| `cut` | Discretiza por rangos definidos | Clasificar por reglas de negocio |
| `qcut` | Discretiza por cuantiles | Crear grupos de percentiles de igual tamaño |

---
## Práctica

Usá el siguiente dataset de empleados de una empresa de logística para resolver los ejercicios.

In [ ]:
import pandas as pd
import numpy as np

data = {
    'empleado':    ['Valeria Ríos', 'Marcos Díaz', 'Lucía Torres', 'Pablo Sosa',
                    'Valeria Ríos', 'Marcos Díaz', 'Lucía Torres', 'Pablo Sosa',
                    'Valeria Ríos', 'Marcos Díaz', 'Lucía Torres', 'Pablo Sosa'],
    'sector':      ['Depósito', 'Distribución', 'Depósito', 'Distribución',
                    'Depósito', 'Distribución', 'Depósito', 'Distribución',
                    'Depósito', 'Distribución', 'Depósito', 'Distribución'],
    'turno':       ['Mañana', 'Tarde', 'Noche', 'Mañana',
                    'Tarde', 'Noche', 'Mañana', 'Tarde',
                    'Noche', 'Mañana', 'Tarde', 'Noche'],
    'categoria':   ['Junior', 'Senior', 'Mid', 'Junior',
                    'Junior', 'Senior', 'Mid', 'Junior',
                    'Mid', 'Senior', 'Mid', 'Senior'],
    'paquetes':    [320, 580, 410, 290, 350, 610, 390, 270, 340, 595, 420, 310],
    'errores':     [  8,   3,   5,  12,   6,   2,   7,  14,   9,   4,   6,  11],
    'horas':       [  8,   8,   7,   8,   8,   9,   8,   7,   7,   8,   8,   9]
}

df_log = pd.DataFrame(data)
df_log

,empleado,sector,turno,categoria,paquetes,errores,horas
0,Valeria Ríos,Depósito,Mañana,Junior,320,8,8
1,Marcos Díaz,Distribución,Tarde,Senior,580,3,8
2,Lucía Torres,Depósito,Noche,Mid,410,5,7
3,Pablo Sosa,Distribución,Mañana,Junior,290,12,8
4,Valeria Ríos,Depósito,Tarde,Junior,350,6,8
5,Marcos Díaz,Distribución,Noche,Senior,610,2,9
6,Lucía Torres,Depósito,Mañana,Mid,390,7,8
7,Pablo Sosa,Distribución,Tarde,Junior,270,14,7
8,Valeria Ríos,Depósito,Noche,Mid,340,9,7
9,Marcos Díaz,Distribución,Mañana,Senior,595,4,8


### Ejercicio 1 — `map`

a) Usá `map` para agregar una columna `'turno_cod'` que convierta el turno a número: Mañana=1, Tarde=2, Noche=3.

b) Usá `map` con una función lambda para agregar una columna `'productividad'` que calcule los paquetes por hora (`paquetes / horas`), redondeado a 1 decimal.

In [ ]:
# a) map con diccionario

turno_a_num = {'Mañana': 1, 'Tarde': 2, 'Noche': 3}
df_log['turno_cod'] = df_log['turno'].map(turno_a_num)
df_log[['turno', 'turno_cod']].drop_duplicates().sort_values('turno_cod')


,turno,turno_cod
0,Mañana,1
1,Tarde,2
2,Noche,3


In [ ]:
# b) map con lambda

df_log['productividad'] = (df_log['paquetes'] / df_log['horas']).map(lambda x: round(x, 1))
df_log[['empleado', 'paquetes', 'horas', 'productividad']]


,empleado,paquetes,horas,productividad
0,Valeria Ríos,320,8,40.0
1,Marcos Díaz,580,8,72.5
2,Lucía Torres,410,7,58.6
3,Pablo Sosa,290,8,36.2
4,Valeria Ríos,350,8,43.8
5,Marcos Díaz,610,9,67.8
6,Lucía Torres,390,8,48.8
7,Pablo Sosa,270,7,38.6
8,Valeria Ríos,340,7,48.6
9,Marcos Díaz,595,8,74.4


### Ejercicio 2 — `transform`

a) Agregá una columna `'media_paquetes_sector'` con la media de paquetes del sector al que pertenece cada empleado.

b) Agregá una columna booleana `'sobre_media_sector'` que indique si esa fila supera la media de su sector.

c) Calculá qué porcentaje representa cada registro sobre el total de paquetes de su empleado. Guardalo en `'pct_del_empleado'`.

In [ ]:
# a)

df_log['media_paquetes_sector'] = df_log.groupby('sector')['paquetes'].transform('mean').round(1)
df_log[['empleado', 'sector', 'paquetes', 'media_paquetes_sector']]


,empleado,sector,paquetes,media_paquetes_sector
0,Valeria Ríos,Depósito,320,371.7
1,Marcos Díaz,Distribución,580,442.5
2,Lucía Torres,Depósito,410,371.7
3,Pablo Sosa,Distribución,290,442.5
4,Valeria Ríos,Depósito,350,371.7
5,Marcos Díaz,Distribución,610,442.5
6,Lucía Torres,Depósito,390,371.7
7,Pablo Sosa,Distribución,270,442.5
8,Valeria Ríos,Depósito,340,371.7
9,Marcos Díaz,Distribución,595,442.5


In [ ]:
# b)

df_log['sobre_media_sector'] = df_log['paquetes'] > df_log['media_paquetes_sector']
df_log[['empleado', 'sector', 'paquetes', 'media_paquetes_sector', 'sobre_media_sector']]


,empleado,sector,paquetes,media_paquetes_sector,sobre_media_sector
0,Valeria Ríos,Depósito,320,371.7,False
1,Marcos Díaz,Distribución,580,442.5,True
2,Lucía Torres,Depósito,410,371.7,True
3,Pablo Sosa,Distribución,290,442.5,False
4,Valeria Ríos,Depósito,350,371.7,False
5,Marcos Díaz,Distribución,610,442.5,True
6,Lucía Torres,Depósito,390,371.7,True
7,Pablo Sosa,Distribución,270,442.5,False
8,Valeria Ríos,Depósito,340,371.7,False
9,Marcos Díaz,Distribución,595,442.5,True


In [ ]:
# c)

total_por_empleado = df_log.groupby('empleado')['paquetes'].transform('sum')
df_log['pct_del_empleado'] = (df_log['paquetes'] / total_por_empleado * 100).round(1)
df_log[['empleado', 'paquetes', 'pct_del_empleado']]


,empleado,paquetes,pct_del_empleado
0,Valeria Ríos,320,31.7
1,Marcos Díaz,580,32.5
2,Lucía Torres,410,33.6
3,Pablo Sosa,290,33.3
4,Valeria Ríos,350,34.7
5,Marcos Díaz,610,34.2
6,Lucía Torres,390,32.0
7,Pablo Sosa,270,31.0
8,Valeria Ríos,340,33.7
9,Marcos Díaz,595,33.3


### Ejercicio 3 — `pivot_table`

a) Creá una tabla dinámica que muestre la cantidad total de paquetes por empleado y turno.

b) Agregá totales por fila y columna con `margins=True`.

c) Creá una segunda tabla que muestre el promedio de errores por sector y categoría.

In [ ]:
# a)

pd.pivot_table(
    df_log,
    values='paquetes',
    index='empleado',
    columns='turno',
    aggfunc='sum',
    fill_value=0
)


turno,Mañana,Noche,Tarde
empleado,,,
Lucía Torres,390,410,420
Marcos Díaz,595,610,580
Pablo Sosa,290,310,270
Valeria Ríos,320,340,350


In [ ]:
# b)

pd.pivot_table(
    df_log,
    values='paquetes',
    index='empleado',
    columns='turno',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)


turno,Mañana,Noche,Tarde,Total
empleado,,,,
Lucía Torres,390,410,420,1220
Marcos Díaz,595,610,580,1785
Pablo Sosa,290,310,270,870
Valeria Ríos,320,340,350,1010
Total,1595,1670,1620,4885


In [ ]:
# c)

pd.pivot_table(
    df_log,
    values='errores',
    index='sector',
    columns='categoria',
    aggfunc='mean',
    fill_value=0
)


categoria,Junior,Mid,Senior
sector,,,
Depósito,7.0,6.75,0.0
Distribución,13.0,0.00,5.0


### Ejercicio 4 — `cut` y `qcut`

a) Usá `cut` para clasificar los paquetes en tres rangos con etiquetas: `'Bajo'` (0-349), `'Medio'` (350-499), `'Alto'` (500 en adelante). Guardalo en `'nivel_produccion'`.

b) Usá `qcut` para dividir los errores en dos grupos de igual tamaño: `'Pocos errores'` y `'Muchos errores'`. Guardalo en `'nivel_error'`.

c) Combiná los resultados: creá una `pivot_table` que muestre el promedio de paquetes cruzando `'nivel_produccion'` con `'sector'`.

In [ ]:
# a)

df_log['nivel_produccion'] = pd.cut(
    df_log['paquetes'],
    bins=[0, 350, 500, float('inf')],
    labels=['Bajo', 'Medio', 'Alto'],
    right=False
)
df_log[['empleado', 'paquetes', 'nivel_produccion']]


,empleado,paquetes,nivel_produccion
0,Valeria Ríos,320,Bajo
1,Marcos Díaz,580,Alto
2,Lucía Torres,410,Medio
3,Pablo Sosa,290,Bajo
4,Valeria Ríos,350,Medio
5,Marcos Díaz,610,Alto
6,Lucía Torres,390,Medio
7,Pablo Sosa,270,Bajo
8,Valeria Ríos,340,Bajo
9,Marcos Díaz,595,Alto


In [ ]:
# b)

df_log['nivel_error'] = pd.qcut(
    df_log['errores'],
    q=2,
    labels=['Pocos errores', 'Muchos errores']
)
df_log[['empleado', 'errores', 'nivel_error']]


,empleado,errores,nivel_error
0,Valeria Ríos,8,Muchos errores
1,Marcos Díaz,3,Pocos errores
2,Lucía Torres,5,Pocos errores
3,Pablo Sosa,12,Muchos errores
4,Valeria Ríos,6,Pocos errores
5,Marcos Díaz,2,Pocos errores
6,Lucía Torres,7,Muchos errores
7,Pablo Sosa,14,Muchos errores
8,Valeria Ríos,9,Muchos errores
9,Marcos Díaz,4,Pocos errores


In [ ]:
# c)

pd.pivot_table(
    df_log,
    values='paquetes',
    index='nivel_produccion',
    columns='sector',
    aggfunc='mean',
    fill_value=0
)


sector,Depósito,Distribución
nivel_produccion,,
Bajo,330.0,290.0
Medio,392.5,0.0
Alto,0.0,595.0


### Ejercicio integrador

Usando todas las herramientas del notebook, respondé las siguientes preguntas sobre el dataset de logística:

1. ¿Cuál es el turno con mayor productividad promedio (paquetes por hora)?
2. ¿Qué empleado tiene la mayor proporción de sus registros por encima de la media de su sector?
3. ¿En qué combinación sector + categoría se producen más errores en promedio?
4. Clasificá a cada empleado en un cuartil de productividad. ¿Hay algún empleado que esté siempre en el mismo cuartil?

In [ ]:
# 1.

df_log.groupby('turno')['productividad'].mean().sort_values(ascending=False)


turno
Noche     52.35
Tarde     51.85
Mañana    49.85
Name: productividad, dtype: float64

In [ ]:
# 2.

df_log.groupby('empleado')['sobre_media_sector'].mean().sort_values(ascending=False)


empleado
Lucía Torres    1.0
Marcos Díaz     1.0
Pablo Sosa      0.0
Valeria Ríos    0.0
Name: sobre_media_sector, dtype: float64

In [ ]:
# 3.

df_log.groupby(['sector', 'categoria'])['errores'].mean().sort_values(ascending=False)


sector        categoria
Distribución  Junior       13.00
Depósito      Junior        7.00
              Mid           6.75
Distribución  Senior        5.00
Name: errores, dtype: float64

In [ ]:
# 4.

df_log['cuartil_prod'] = pd.qcut(df_log['productividad'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_log.groupby(['empleado', 'cuartil_prod']).size().reset_index(name='registros')


,empleado,cuartil_prod,registros
0,Lucía Torres,Q3,3
1,Marcos Díaz,Q4,3
2,Pablo Sosa,Q1,3
3,Valeria Ríos,Q2,3
